# IANÃ: Estruturador Clínico Padrão SOAP (Nível 01)
## Fase: Filtragem de Doenças Alvo e Preparação de Corpus

**Objetivo:** Isolar prontuários da base MIMIC-IV contendo os diagnósticos de interesse (HIV, Sífilis e Tuberculose) para posterior divisão de sentenças e anotação manual.
**Pipeline de Referência:** Metodologia de Oliveira et al. (2024) / Nível I e II.

In [10]:
import polars as pl

# 1. Caminho EXATO com a subpasta do MIMIC incluída
caminho_mimic = '../data/mimic-iv-note-deidentified-free-text-clinical-notes-2.2/note_parquet/discharge.parquet'

print("⏳ Carregando a base completa do MIMIC-IV...")
df = pl.read_parquet(caminho_mimic)
print(f"✅ Base carregada! Total de prontuários originais: {df.height:,}")

# 2. Criando o Filtro (Regex) para as 3 Doenças da Samantha (Em Inglês)
regex_hiv = r'(?i)\b(hiv|aids|human immunodeficiency virus)\b'
regex_sifilis = r'(?i)\b(syphilis|treponema pallidum)\b'
regex_tb = r'(?i)\b(tuberculosis|mycobacterium tuberculosis)\b'

regex_alvo = f"({regex_hiv}|{regex_sifilis}|{regex_tb})"

# 3. Filtrando
print("🔍 Filtrando pacientes com HIV, Sífilis ou Tuberculose...")
df_soap = df.filter(
    pl.col('text').str.contains(regex_alvo)
)

print(f"🎯 Filtragem concluída! Encontramos {df_soap.height:,} prontuários alvo.")

# 4. Salvando na pasta data principal (fora daquela pasta gigante)
caminho_saida = '../data/dataset_soap_hiv_tb_sifilis.parquet'
df_soap.write_parquet(caminho_saida)
print(f"💾 Base otimizada salva em: {caminho_saida}")

⏳ Carregando a base completa do MIMIC-IV...
✅ Base carregada! Total de prontuários originais: 331,793
🔍 Filtrando pacientes com HIV, Sífilis ou Tuberculose...
🎯 Filtragem concluída! Encontramos 24,828 prontuários alvo.
💾 Base otimizada salva em: ../data/dataset_soap_hiv_tb_sifilis.parquet


### ✂️ Nível 01: Divisão de Sentenças (Sentence Splitting)

**Resumo da Filtragem:** A extração reduziu o volume total de dados em aproximadamente 92%, isolando **24.828 prontuários** que contêm diagnósticos confirmados de HIV, Sífilis ou Tuberculose. 

**O Próximo Desafio (Segmentação):**
Para que o classificador de Inteligência Artificial (ClinicalBERT/Gemma) consiga alocar as informações nas categorias **S.O.A.P.**, ele não pode ler o documento inteiro de uma vez. Ele precisa analisar o texto **frase por frase**.

Abaixo, iniciaremos a etapa de pré-processamento de NLP (Processamento de Linguagem Natural). O objetivo é pegar os blocos de texto denso (frequentemente sem pontuação clara ou cheios de abreviações médicas) e dividi-los em sentenças isoladas. Isso criará o *Corpus de Evoluções Clínicas Segmentadas* estruturado.

In [12]:
import polars as pl
import pysbd
from tqdm import tqdm

print("⏳ Carregando o modelo PySBD (Focado em textos médicos)...")
# Inicializa o segmentador para inglês
seg = pysbd.Segmenter(language="en", clean=False)

# 1. Carregando a nossa base filtrada
caminho_base = '../data/dataset_soap_hiv_tb_sifilis.parquet'
df_alvo = pl.read_parquet(caminho_base)

# Pegamos uma amostra de 1.000 prontuários para gerar o corpus de anotação
df_amostra = df_alvo.head(1000)
print(f"✅ Amostra carregada: {df_amostra.height} prontuários.")

textos = df_amostra['text'].to_list()
subject_ids = df_amostra['subject_id'].to_list()
note_ids = df_amostra['note_id'].to_list()

# 2. Processando as sentenças
print("🔪 Fatiando os prontuários em sentenças (Padrão SOAP)...")
dados_segmentados = []

# tqdm cria a barra de progresso no terminal
for i in tqdm(range(len(textos))):
    texto_medico = textos[i]
    
    # O PySBD corta o texto entendendo abreviações médicas (ex: Dr., p.o., mg.)
    frases = seg.segment(texto_medico)
    
    for frase in frases:
        frase_limpa = frase.strip()
        # Filtramos sujeiras, quebras de linha ou frases minúsculas
        if len(frase_limpa) > 10: 
            dados_segmentados.append({
                "subject_id": subject_ids[i],
                "note_id": note_ids[i],
                "sentence": frase_limpa
            })

# 3. Convertendo para Polars e salvando em CSV para o Label Studio
df_sentencas = pl.DataFrame(dados_segmentados)

caminho_corpus = '../data/corpus_segmentado_amostra_soap.csv'
df_sentencas.write_csv(caminho_corpus)

print(f"\n🎯 SUCESSO! Transformamos 1.000 prontuários em {df_sentencas.height:,} frases isoladas.")
print(f"💾 Arquivo pronto para a ferramenta de anotação salvo em: {caminho_corpus}")

⏳ Carregando o modelo PySBD (Focado em textos médicos)...
✅ Amostra carregada: 1000 prontuários.
🔪 Fatiando os prontuários em sentenças (Padrão SOAP)...


100%|██████████| 1000/1000 [02:06<00:00,  7.90it/s]



🎯 SUCESSO! Transformamos 1.000 prontuários em 303,142 frases isoladas.
💾 Arquivo pronto para a ferramenta de anotação salvo em: ../data/corpus_segmentado_amostra_soap.csv


### 📊 Resultado da Segmentação Clínica (Sentence Boundary Disambiguation)

**Métrica de Extração:** Uma amostra inicial de 1.000 prontuários foi processada pelo modelo de linguagem `PySBD`, resultando em um corpus massivo de **303.142 sentenças isoladas**.

**Análise do Resultado:**
* **Densidade Clínica:** O alto volume reflete a complexidade das Notas de Alta (Discharge Summaries), com uma média de ~303 frases redigidas por documento.
* **Superação de Meta (Fase 02):** A metodologia de referência previa a necessidade de aproximadamente 100.000 sentenças para a anotação manual do Padrão Ouro. Com apenas 4% da base filtrada (~1.000 de 24.828), já geramos um volume 3x maior que o necessário para treinar o classificador clínico.

**Status:** O arquivo `corpus_segmentado_amostra_soap.csv` constitui o **Corpus Bruto**. Ele já pode ser exportado para plataformas de rotulação (ex: Label Studio) para que a equipe médica anote as classes estruturais (Subjetivo, Objetivo, Avaliação e Plano).

### 🛡️ Arquitetura de Validação Pydantic (Preparação para a DGX)

Com as sentenças segmentadas prontas, o próximo passo é garantir que o modelo de linguagem (LLM) que rodará na DGX retorne os dados em um formato estritamente padronizado e compatível com o **SemClinBR**.

Abaixo, definimos o Schema (Tipagem Forte) usando `Pydantic`. Ele atua como uma "camisa de força" para a IA:
1. **Enums:** Travam as opções. A IA não pode inventar tags novas.
2. **Tipagem:** Garante que listas sejam listas e que notas de confiança sejam números de 0 a 1.
3. **Autocorreção:** Se a IA alucinar o formato, o Pydantic gera um erro estruturado (`ValidationError`) que servirá de gatilho para o *Retry* automático do Agente.

*Nota: As tags atuais são *dummies* (marcadores) e serão substituídas pela lista oficial de 20 tags assim que definidas pela equipe médica (Samantha).*

In [3]:
from pydantic import BaseModel, Field, ValidationError
from typing import List
from enum import Enum
import json

# ==========================================
# 1. AS REGRAS DO JOGO (Prompt e Tags Oficiais)
# ==========================================

class CategoriaSOAP(str, Enum):
    SUBJETIVO = "Subjetivo"
    OBJETIVO = "Objetivo"
    AVALIACAO = "Avaliacao"
    PLANO = "Plano"

class TagEntidade(str, Enum):
    # Tags oficiais extraídas das planilhas da equipe médica
    ABBREVIATION = "Abbreviation"
    AGE_GROUP = "Age Group"
    ANTIBIOTIC = "Antibiotic"
    BODY_PART = "Body Part, Organ, or Organ Component"
    BODY_SUBSTANCE = "Body Substance"
    CLINICAL_ATTRIBUTE = "Clinical Attribute"
    DIAGNOSTIC_PROCEDURE = "Diagnostic Procedure"
    DISEASE_SYNDROME = "Disease or Syndrome"
    FAMILY_GROUP = "Family Group"
    FINDING = "Finding"
    HEALTH_CARE_ACTIVITY = "Health Care Activity"
    LABORATORY_PROCEDURE = "Laboratory Procedure"
    LABORATORY_TEST_RESULT = "Laboratory or Test Result"
    NEGATION = "Negation"
    ORGANISM_ATTRIBUTE = "Organism Attribute"
    PATHOLOGIC_FUNCTION = "Pathologic Function"
    PHARMACOLOGIC_SUBSTANCE = "Pharmacologic Substance"
    POPULATION_GROUP = "Population Group"
    SIGN_SYMPTOM = "Sign or Symptom"
    TEMPORAL_CONCEPT = "Temporal Concept"
    THERAPEUTIC_PROCEDURE = "Therapeutic or Preventive Procedure"
    VIRUS = "Virus"

# ==========================================
# 2. OS MODELOS STRICT (A "Catraca")
# ==========================================

class EntidadeClinica(BaseModel):
    termo: str = Field(..., description="Trecho exato do texto (ex: 'VDRL NR')")
    tag: TagEntidade = Field(..., description="Classificação baseada no SemClinBR")

class ExtracaoSOAP(BaseModel):
    sentenca_original: str = Field(..., description="A frase isolada pelo modelo PySBD")
    
    # 🧠 Aqui mora o coração da Engenharia de Prompt do SOAP
    classificacao_soap: CategoriaSOAP = Field(
        ..., 
        description="""Classifique a sentença estritamente em uma destas categorias:
        - Subjetivo: Relatos do paciente, queixas, sintomas sentidos e histórico familiar.
        - Objetivo: Sinais vitais, resultados de exames (laboratório/imagem) e exame físico.
        - Avaliacao: Diagnósticos, hipóteses médicas, evolução clínica do paciente.
        - Plano: Tratamentos, medicamentos prescritos, procedimentos, próximos passos e condutas."""
    )
    
    entidades_encontradas: List[EntidadeClinica] = Field(default_factory=list)
    confianca_modelo: float = Field(ge=0.0, le=1.0)

print("✅ Arquitetura Pydantic atualizada com as tags oficiais da equipe médica e regras SOAP!")

✅ Arquitetura Pydantic atualizada com as tags oficiais da equipe médica e regras SOAP!


In [4]:
import polars as pl

# Quando seu amigo mandar o arquivo hoje à noite, você coloca o nome do arquivo aqui:
arquivo_curadoria = 'data/termos_revisados_equipe.csv' 

try:
    print(f"⏳ Carregando a curadoria médica do arquivo: {arquivo_curadoria}")
    
    # Lendo o arquivo (ajuste read_csv ou read_excel dependendo do que ele mandar)
    df_termos = pl.read_csv(arquivo_curadoria)
    
    # O Pulo do Gato: Filtrando apenas as linhas onde a coluna de marcação for 1
    # Nota: Você precisará ajustar 'nome_da_coluna_de_nota' para o nome exato que seu amigo usar (ex: 'len', 'motivo', ou 'relevante')
    coluna_nota = 'len' 
    
    df_termos_filtrados = df_termos.filter(pl.col(coluna_nota) == 1)
    
    # Criando uma lista limpa dos termos médicos (mention) para usar como vocabulário da IA
    vocabulario_medico = df_termos_filtrados['mention'].to_list()
    
    print(f"🎯 Sucesso! De {df_termos.height} termos totais, filtramos {len(vocabulario_medico)} termos válidos (Nota 1).")
    print("\nExemplo dos termos aprovados pela equipe:")
    print(vocabulario_medico[:10])
    
except FileNotFoundError:
    print("⏳ Célula engatilhada: Aguardando o envio do arquivo final pelo colega da equipe.")

⏳ Carregando a curadoria médica do arquivo: data/termos_revisados_equipe.csv
⏳ Célula engatilhada: Aguardando o envio do arquivo final pelo colega da equipe.
